In [ ]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

from __future__ import annotations

import importlib
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_fasta_snapshot as ncbi_fasta_snapshot_module
import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_fasta_snapshot_module = importlib.reload(ncbi_fasta_snapshot_module)
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
resolve_ncbi_protein_fasta_snapshot = (
    ncbi_fasta_snapshot_module.resolve_ncbi_protein_fasta_snapshot
)

from src.pago_pipeline.storage import sha256_of_file

In [ ]:
# =============================================================================
# CELL 2 — Load environment and resolve project root
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your project configuration at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# =============================================================================
# CELL 3 — Define FASTA snapshot configuration
# =============================================================================

XML_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "01-raw" / "protein_xml_snapshots"
)
METADATA_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "02-intermediate" / "protein_metadata_csv"
)
FASTA_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "02-intermediate" / "protein_fasta"
)
FASTA_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
SEQUENCE_LINE_WIDTH = 60
UPDATE_LATEST_DIRECTORY = True

print(f"XML snapshot root directory: {XML_SNAPSHOT_ROOT_DIRECTORY}")
print(f"Metadata snapshot root directory: {METADATA_SNAPSHOT_ROOT_DIRECTORY}")
print(f"FASTA snapshot root directory: {FASTA_SNAPSHOT_ROOT_DIRECTORY}")
print(f"FASTA snapshot mode: {FASTA_SNAPSHOT_MODE}")
print(f"Sequence line width: {SEQUENCE_LINE_WIDTH}")

In [ ]:
# =============================================================================
# CELL 4 — Resolve active FASTA snapshot
# =============================================================================

fasta_snapshot_payload = resolve_ncbi_protein_fasta_snapshot(
    snapshot_mode=FASTA_SNAPSHOT_MODE,
    snapshot_root_directory=FASTA_SNAPSHOT_ROOT_DIRECTORY,
    source_metadata_snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
    sequence_line_width=SEQUENCE_LINE_WIDTH,
    update_latest_directory=UPDATE_LATEST_DIRECTORY,
)

protein_fasta_snapshot_directory = fasta_snapshot_payload["snapshot_directory"]
protein_fasta_manifest_file_path = fasta_snapshot_payload["manifest_file_path"]
protein_fasta_file_path = fasta_snapshot_payload["fasta_file_path"]
protein_fasta_manifest_payload = fasta_snapshot_payload["manifest"]
protein_fasta_output_directory = protein_fasta_snapshot_directory

protein_fasta_file_sha256 = sha256_of_file(input_file_path=protein_fasta_file_path)
protein_fasta_manifest_file_sha256 = sha256_of_file(
    input_file_path=protein_fasta_manifest_file_path,
)

source_metadata_snapshot_relative_path = protein_fasta_manifest_payload.get(
    "source_metadata_snapshot_relative_path"
)
if (
    not isinstance(source_metadata_snapshot_relative_path, str)
    or not source_metadata_snapshot_relative_path
):
    raise RuntimeError(
        "FASTA snapshot manifest is missing source_metadata_snapshot_relative_path."
    )

source_metadata_csv_file_name = protein_fasta_manifest_payload.get(
    "source_metadata_csv_file_name"
)
if not isinstance(source_metadata_csv_file_name, str) or not source_metadata_csv_file_name:
    raise RuntimeError(
        "FASTA snapshot manifest is missing source_metadata_csv_file_name."
    )

source_metadata_snapshot_directory = (
    METADATA_SNAPSHOT_ROOT_DIRECTORY / source_metadata_snapshot_relative_path
)
source_metadata_csv_file_path = (
    source_metadata_snapshot_directory / source_metadata_csv_file_name
)

source_xml_snapshot_relative_path = protein_fasta_manifest_payload.get(
    "source_xml_snapshot_relative_path"
)
if not isinstance(source_xml_snapshot_relative_path, str) or not source_xml_snapshot_relative_path:
    raise RuntimeError(
        "FASTA snapshot manifest is missing source_xml_snapshot_relative_path."
    )

source_xml_snapshot_directory = (
    XML_SNAPSHOT_ROOT_DIRECTORY / source_xml_snapshot_relative_path
)

print("Resolved FASTA snapshot successfully.")
print(f"FASTA snapshot directory: {protein_fasta_snapshot_directory}")
print(f"FASTA file path: {protein_fasta_file_path}")
print(f"FASTA manifest file path: {protein_fasta_manifest_file_path}")
print(f"Source metadata CSV file path: {source_metadata_csv_file_path}")

In [ ]:
# =============================================================================
# CELL 5 — Print FASTA snapshot summary
# =============================================================================

print("Protein FASTA snapshot is ready.")
print(
    f"Snapshot created at UTC: {protein_fasta_manifest_payload['snapshot_created_at_utc']}"
)
print(f"Rows read from source metadata CSV: {protein_fasta_manifest_payload['row_count']}")
print(f"FASTA records written: {protein_fasta_manifest_payload['fasta_record_count']}")
print(
    "Rows skipped because the AA sequence column was empty: "
    f"{protein_fasta_manifest_payload['skipped_missing_sequence_count']}"
)
print(
    "Source metadata snapshot relative path: "
    f"{protein_fasta_manifest_payload['source_metadata_snapshot_relative_path']}"
)

In [ ]:
# =============================================================================
# CELL 6 — Print FASTA artifact summary
# =============================================================================

print("Persisted FASTA snapshot artifacts:")
print(f"FASTA snapshot directory: {protein_fasta_snapshot_directory}")
print(f"FASTA file path: {protein_fasta_file_path}")
print(f"FASTA file SHA-256: {protein_fasta_file_sha256}")
print(f"FASTA manifest file path: {protein_fasta_manifest_file_path}")
print(f"FASTA manifest SHA-256: {protein_fasta_manifest_file_sha256}")
print(f"Source metadata snapshot directory: {source_metadata_snapshot_directory}")
print(f"Source metadata CSV file path: {source_metadata_csv_file_path}")
print(
    "Source metadata CSV SHA-256: "
    f"{protein_fasta_manifest_payload['source_metadata_csv_file_sha256']}"
)
print(
    "Source metadata manifest SHA-256: "
    f"{protein_fasta_manifest_payload['source_metadata_manifest_sha256']}"
)

In [ ]:
# =============================================================================
# CELL 7 — Inspect metadata columns used for FASTA export
# =============================================================================

fasta_source_columns = [
    "protein_uid",
    "gbseq__accession_version",
    "gbseq__length",
    "gbseq__organism",
    "gbseq__definition",
    "gbseq__sequence",
]
preview_row_limit = 3

fasta_source_preview_dataframe = pd.read_csv(
    source_metadata_csv_file_path,
    usecols=fasta_source_columns,
    nrows=preview_row_limit,
)

fasta_source_preview_dataframe

In [ ]:
# =============================================================================
# CELL 8 — Preview FASTA output
# =============================================================================

preview_line_limit = 8
fasta_preview_lines: list[str] = []

with protein_fasta_file_path.open("r", encoding="utf-8") as fasta_file_handle:
    for _ in range(preview_line_limit):
        fasta_line = fasta_file_handle.readline()
        if not fasta_line:
            break
        fasta_preview_lines.append(fasta_line.rstrip("\n"))

print("Preview of the exported multi-FASTA file:")
print("\n".join(fasta_preview_lines))

In [ ]:
# =============================================================================
# CELL 9 — Print FASTA provenance summary
# =============================================================================

print("FASTA provenance summary:")
print(
    "Source metadata snapshot relative path: "
    f"{protein_fasta_manifest_payload['source_metadata_snapshot_relative_path']}"
)
print(
    "Source XML snapshot relative path: "
    f"{protein_fasta_manifest_payload['source_xml_snapshot_relative_path']}"
)
print(
    "Source XML snapshot directory name: "
    f"{protein_fasta_manifest_payload['source_xml_snapshot_directory_name']}"
)
print(f"Search query: {protein_fasta_manifest_payload['search_query']}")
print(f"Translated query: {protein_fasta_manifest_payload['translated_query']}")

In [ ]:
# =============================================================================
# CELL 10 — Expose downstream variables
# =============================================================================

print("Variables exposed for downstream notebooks:")
print("- protein_fasta_snapshot_directory")
print("- protein_fasta_output_directory")
print("- protein_fasta_file_path")
print("- protein_fasta_manifest_file_path")
print("- protein_fasta_manifest_payload")
print("- source_metadata_snapshot_directory")
print("- source_metadata_csv_file_path")
print("- source_xml_snapshot_directory")